<a id="top"></a>

# Lab L1004: build the ReAct agent graph (termination)

```yaml
title: "Lab L1004: build the ReAct agent graph (termination)"
keywords: ReAct agent, StateGraph, add_messages, ToolNode, route, conditional edge, back-edge, cycle, natural termination, recursion_limit, GraphRecursionError, FakeModel, offline, lab
estimated duration: 40
```

> **Lesson:** L10. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L10/objectives.md).
> Solutions copy — every `# TODO` filled in. Your copy: `L1004_lab_empty.ipynb`. **No API key: pure
> Python.** You wire the graph against a *scripted stub model*, so every run comes out identical (same
> offline trick as the [L1003 demo](L1003_lecture.ipynb)).
>
> **You'll walk out able to** build a ReAct agent as a cyclic `StateGraph` — `agent` node, prebuilt
> `ToolNode`, a `route` conditional edge, and the **back-edge** — and say exactly how it stops:
> naturally when `route` returns `END`, or on the `recursion_limit` cap when it won't.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — Write route (the conditional edge)](#2-problem-1--write-route-the-conditional-edge)
- [3. Problem 2 — Wire the graph: build_agent](#3-problem-2--wire-the-graph-build_agent)
- [4. Problem 3 — Drive it: natural termination](#4-problem-3--drive-it-natural-termination)
- [5. Problem 4 — Drive it: the recursion_limit catches a runaway](#5-problem-4--drive-it-the-recursion_limit-catches-a-runaway)
- [6. Problem 5 — The message-history invariant (written)](#6-problem-5--the-message-history-invariant-written)
- [7. Self-check](#7-self-check)

## 1. Setup

All handed to you: the `calculator` + `lookup` tools (and the `TOOLS` list), a scripted **`FakeModel`**
from `common` that fakes a LangChain chat model — `.bind_tools(...)`, then an awaited
`.ainvoke(messages)` hands back the next scripted `AIMessage` — and the **`AgentState`** with its
`add_messages` reducer. **Your job: the graph.** Two functions — `route`, then `build_agent`.

In [ ]:
from typing import Annotated, Any, TypedDict

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)


def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression, or raise ValueError on non-arithmetic input."""
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    return str(eval(expression))


POPULATIONS = {"tokyo": "37000000", "lagos": "15000000", "paris": "11000000"}


def lookup(city: str) -> str:
    """Return a city's population, or raise KeyError if it isn't in the table."""
    key = city.strip().lower()
    if key not in POPULATIONS:
        raise KeyError(f"no population on file for {city!r}")
    return POPULATIONS[key]


# The tool list handed to both the model (bind_tools) and the ToolNode.
TOOLS = [calculator, lookup]


class AgentState(TypedDict):
    """The agent's state. The `add_messages` reducer APPENDS each node's messages to the
    running conversation (in L03 a returned key OVERWROTE it) -- that is why the history grows."""

    messages: Annotated[list[BaseMessage], add_messages]

[↑ Back to top](#top)

## 2. Problem 1 — Write `route` (the conditional edge)

Warm-up before the whole graph. `route(state)` is your L05 conditional edge, and it only ever looks at
the **last** message. `AIMessage` with non-empty `.tool_calls`? Return `"tools"` — go run them.
Anything else? Return `END` — the model sent plain text, and that's *natural termination*.

In [2]:
def route(state: AgentState) -> str:
    """Conditional edge: got tool calls? -> the 'tools' node; else -> END (natural termination)."""
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END


# Quick checks:
print(route({"messages": [tool_reply(tool_call("c1", "lookup", {"city": "Paris"}))]}))  # -> tools
print(route({"messages": [text_reply("all done")]}))  # -> __end__ (natural termination)

tools
__end__


[↑ Back to top](#top)

## 3. Problem 2 — Wire the graph: `build_agent`

Now the whole thing. First `agent_node` (an `async def`): bind `TOOLS`,
`await ... .ainvoke(state["messages"])`, return `{"messages": [reply]}`. Then wire and compile —
`StateGraph(AgentState)`, add the `agent` node and a `ToolNode(TOOLS)` `tools` node,
`set_entry_point("agent")`, `add_conditional_edges("agent", route, {"tools": "tools", END: END})`,
and the one edge that makes it a loop: the **back-edge**, `add_edge("tools", "agent")`.

In [3]:
def build_agent(model: Any) -> Any:
    """Wire and compile the ReAct graph: agent node + ToolNode + conditional route + back-edge."""

    async def agent_node(state: AgentState) -> dict[str, list[BaseMessage]]:
        """Call the tool-bound model on the conversation; return its one reply to be appended."""
        reply = await model.bind_tools(TOOLS).ainvoke(state["messages"])
        return {"messages": [reply]}

    builder = StateGraph(AgentState)
    builder.add_node("agent", agent_node)
    builder.add_node("tools", ToolNode(TOOLS))
    builder.set_entry_point("agent")
    builder.add_conditional_edges("agent", route, {"tools": "tools", END: END})
    builder.add_edge("tools", "agent")  # the back-edge -- this is what makes it a cycle
    return builder.compile()


# Confirm the shape: two real nodes and the all-important back-edge.
rendered = build_agent(FakeModel([text_reply("x")])).get_graph()
print("nodes:", list(rendered.nodes))
print(
    "back-edge tools -> agent:",
    ("tools", "agent") in [(e.source, e.target) for e in rendered.edges],
)

nodes: ['__start__', 'agent', 'tools', '__end__']
back-edge tools -> agent: True


[↑ Back to top](#top)

## 4. Problem 3 — Drive it: natural termination

Script a stub that calls `calculator`, then `lookup`, then answers in plain text. Build it,
`await` its `ainvoke`, and prove it stopped on its own: tool sequence `['calculator', 'lookup']`,
and a last message that's a plain-text `AIMessage` with no tool calls — *natural termination*.

In [4]:
happy_script = [
    tool_reply(tool_call("c1", "calculator", {"expression": "36 * 1"})),  # -> 36
    tool_reply(tool_call("c2", "lookup", {"city": "Lagos"})),  # -> 15000000
    text_reply("Lagos has about 15,000,000 people."),
]
graph = build_agent(FakeModel(happy_script))
task = {"messages": [HumanMessage(content="Population of Lagos? Use the tools.")]}
result = await graph.ainvoke(task)

tool_sequence = [
    c["name"] for m in result["messages"] if isinstance(m, AIMessage) for c in m.tool_calls
]
print("tool sequence:", tool_sequence)
print("final answer :", result["messages"][-1].content)
assert tool_sequence == ["calculator", "lookup"]
assert isinstance(result["messages"][-1], AIMessage) and not result["messages"][-1].tool_calls

tool sequence: ['calculator', 'lookup']
final answer : Lagos has about 15,000,000 people.


[↑ Back to top](#top)

## 5. Problem 4 — Drive it: the `recursion_limit` catches a runaway

Now break it on purpose. Script a stub that **never stops** — a stack of distinct `lookup` turns.
`await` an `ainvoke` with `{"recursion_limit": 6}` and watch `GraphRecursionError` fire. That's the
safety net: a cyclic graph with no cap is a runaway waiting to happen — and a tripped cap is a signal
to investigate, not noise to swallow.

In [5]:
# A model that never stops: distinct lookup turns (r0, r1, ...) so add_messages appends each one
# and the cycle actually runs away (repeating the SAME object would be de-duplicated by id).
runaway_script = [tool_reply(tool_call(f"r{i}", "lookup", {"city": "Tokyo"})) for i in range(8)]
runaway_graph = build_agent(FakeModel(runaway_script))

raised = False
try:
    await runaway_graph.ainvoke(
        {"messages": [HumanMessage(content="Keep re-checking Tokyo forever.")]},
        {"recursion_limit": 6},  # halt after 6 node visits
    )
except GraphRecursionError:
    raised = True

print("recursion_limit fired:", raised, " <- the cap caught the runaway, not the model")
assert raised

recursion_limit fired: True  <- the cap caught the runaway, not the model


[↑ Back to top](#top)

## 6. Problem 5 — The message-history invariant (written)

The `agent` node just returned an `AIMessage` with **two** tool calls. Before the next `agent` visit,
what has to be true of the message list — and which **two prebuilt pieces** guarantee it, with zero
bookkeeping from you? (Hint: one's the state's reducer, one's a node.)

_Answer in this cell (double-click to edit)._

**Answer:** Every tool call needs a matching `ToolMessage` (paired by `tool_call_id`) appended
*before* the next model call — two tool calls, two `ToolMessage`s. Two prebuilt pieces guarantee it
with zero bookkeeping from you: the **`add_messages` reducer** on `AgentState` (each node *appends*
to the conversation instead of overwriting it) and the prebuilt **`ToolNode`** (it runs every
requested tool and appends one `ToolMessage` per call). Hand-write that pairing yourself and it
becomes the number-one bug; here it's a graph invariant.

[↑ Back to top](#top)

## 7. Self-check

Diff against `L1004_lab_empty.ipynb`. You're done when: `route` returns `"tools"` for a tool-call
reply and `END` for a text reply; `build_agent` compiles a graph with an `agent` node, a `tools` node,
and a `tools → agent` back-edge; the happy run goes `['calculator', 'lookup']` and stops naturally; the
runaway run trips `GraphRecursionError` at `recursion_limit=6`; and you can name the two prebuilt pieces
that keep the message-history invariant.

[↑ Back to top](#top)